In [1]:
# conda activate dream

library(sva)
library(edgeR)
library(limma)
library(dplyr)
library(data.table)

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

Loading required package: mgcv

Loading required package: nlme

This is mgcv 1.9-3. For overview type 'help("mgcv-package")'.

Loading required package: genefilter

Loading required package: BiocParallel

Loading required package: limma


Attaching package: ‘dplyr’


The following object is masked from ‘package:nlme’:

    collapse


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
batch_ANOVA <- function(pca, sampleinfo, batch_cols) {
    for (i in batch_cols) {
        batch_label <- colnames(sampleinfo)[i]
        batch <- factor(sampleinfo[,i])
        if (length(levels(batch)) > 1) {
            if (batch_label == "Mean_age") {
                batch <- sampleinfo[,i] 
            }
            fit <- lm(pca$x[,2] ~ batch)
            pc_test <- anova(fit)$`Pr(>F)`[1]
            print(paste(batch_label, p.adjust(pc_test, "BH")))
        }
    }
}

In [3]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)

In [4]:
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

batch_cols <- c(
    grep("Mean_age", colnames(sampleinfo)),
    grep("SMCENTER", colnames(sampleinfo)),
    grep("SMNABTCH$", colnames(sampleinfo)), 
    grep("SMGEBTCH$", colnames(sampleinfo)),
    grep("SEX", colnames(sampleinfo)),
    grep("SMTSD", colnames(sampleinfo)),
    grep("DTHHRDY", colnames(sampleinfo))
)

## Starting from TPM QN rd2 data

### Prep data

In [11]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)
expr <- fread("GTEx_cortex_TPM_rd2_SampleNetworks/All_02-16-53/GTEx_cortex_TPM_rd2_All_501_outliers_removed.csv", data.table=FALSE)

In [12]:
sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[match(colnames(expr)[-1], sampleinfo[,1]),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

In [13]:
expr[1:4,1:4]

,datExprT...c.1.skip1..,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y
,<chr>,<dbl>,<dbl>,<dbl>
1,DDX11L1,0.0000000,0.000000,0.000000
2,WASH7P,2.5457173,3.642921,2.463347
3,MIR6859-1,0.0000000,0.000000,0.000000
4,MIR1302-2HG,0.0477403,0.000000,0.000000


In [14]:
# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

[1] 55692     3


In [ ]:
# Remove 0 expression genes
expr_filtered <- expr[rowSums(expr[,-1]) > 0,]
dim(expr_filtered)

rownames(expr_filtered) <- expr_filtered[,1]
dat <- expr_filtered[,-1]

[1] 55434   502

[1] 55434   501

### Check significance of batch variables

In [16]:
# Before ComBat

pca <- prcomp(t(dat), center=TRUE, scale.=TRUE)

In [17]:
batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.00871379213224549"
[1] "SMCENTER 1.30219394070189e-89"
[1] "SMNABTCH 1.98609780199906e-26"
[1] "SMGEBTCH 7.19525783354419e-11"
[1] "SEX 0.0819922841346042"
[1] "SMTSD 4.35780494126276e-95"
[1] "DTHHRDY 0.086935571229318"


### Run ComBat

In [18]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(SMTSD), data=sampleinfo)

batch <- factor(sampleinfo$SMGEBTCH)

expr_combat1 <- sva::ComBat(
    dat,
    batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 31699 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found173batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [19]:
# After ComBat

pca <- prcomp(t(expr_combat1), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.0579844395345969"
[1] "SMCENTER 4.22330678183773e-129"
[1] "SMNABTCH 5.221694493199e-33"
[1] "SMGEBTCH 2.99337082446483e-06"
[1] "SEX 0.063037741487689"
[1] "SMTSD 6.34001343788927e-139"
[1] "DTHHRDY 0.178911747586746"


In [ ]:
fwrite(expr_combat1, file=paste0("data/GTEx_cortex_TPM_QN_rd2_All_501_outliers_removed_ComBat_SMGEBTCH_corrected.csv"), row.names=TRUE)

x being coerced from class: matrix to data.table



## Starting from TPM data

### Prep data

In [5]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)
expr <- fread("GTEx_cortex_TPM_SampleNetworks/All_12-10-19/GTEx_cortex_TPM_All_598_outliers_removed.csv", data.table=FALSE)

In [6]:
sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[match(colnames(expr)[-1], sampleinfo[,1]),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

In [8]:
dim(expr)

[1] 56997   599

In [9]:
# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

[1] 56027     3


In [10]:
# Remove 0 expression genes
expr_filtered <- expr[rowSums(expr[,-1]) > 0,]
dim(expr_filtered)

rownames(expr_filtered) <- expr_filtered[,1]
dat <- expr_filtered[,-1]

[1] 55692   599

### Check significance of batch variables

In [11]:
# Before ComBat

pca <- prcomp(t(dat), center=TRUE, scale.=TRUE)

In [12]:
batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 5.73087922165035e-05"
[1] "SMCENTER 1.09864551376345e-18"
[1] "SMNABTCH 1.18801991938684e-05"
[1] "SMGEBTCH 0.000131627046141871"
[1] "SEX 0.000944171575397628"
[1] "SMTSD 5.16954905487537e-20"
[1] "DTHHRDY 2.81444966255332e-22"


### Run ComBat

In [13]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(SMTSD), data=sampleinfo)

batch <- factor(sampleinfo$SMGEBTCH)

expr_combat1 <- sva::ComBat(
    dat,
    batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 31595 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found190batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [14]:
# After ComBat

pca <- prcomp(t(expr_combat1), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 2.35276273710212e-06"
[1] "SMCENTER 2.10521688721933e-33"
[1] "SMNABTCH 7.41942148808865e-07"
[1] "SMGEBTCH 0.924873845953767"
[1] "SEX 0.000165998435292084"
[1] "SMTSD 2.90259280320896e-35"
[1] "DTHHRDY 1.70134039430311e-19"


In [15]:
fwrite(expr_combat1, file=paste0("data/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected.csv"), row.names=TRUE)

x being coerced from class: matrix to data.table



## Starting from counts data

### Prep data

In [5]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)
expr <- fread("GTEx_cortex_counts_SampleNetworks/All_10-37-43/GTEx_cortex_counts_All_501_outliers_removed.csv", data.table=FALSE)

In [6]:
expr[1:5,1:4]

,datExprT...c.1.skip1..,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y
,<chr>,<int>,<int>,<int>
1,DDX11L1,0,0,0
2,WASH7P,144,164,134
3,MIR6859-1,0,0,0
4,MIR1302-2HG,2,0,0
5,FAM138A,0,0,0


In [7]:
sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[match(colnames(expr)[-1], sampleinfo[,1]),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

In [8]:
# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr, with_ties=FALSE)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

[1] 56027     3


In [9]:
# Remove 0 expression genes
expr_filtered <- expr[rowSums(expr[,-1]) > 0,]
dim(expr_filtered)

[1] 55405   502

In [10]:
total_expr <- colSums(expr_filtered[,-1])
expr_norm <- sweep(expr_filtered[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4

In [11]:
rownames(expr_norm) <- expr_filtered[,1]
dat <- expr_norm
dim(dat)

[1] 55405   501

### Check significance of batch variables

In [12]:
# Before ComBat

pca <- prcomp(t(dat), center=TRUE, scale.=TRUE)

In [13]:
batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 8.61179707381759e-05"
[1] "SMCENTER 7.08262392488317e-30"
[1] "SMNABTCH 1.7963266005222e-09"
[1] "SMGEBTCH 0.0030482337136046"
[1] "SEX 0.00614620873745559"
[1] "SMTSD 2.93392773323101e-31"
[1] "DTHHRDY 9.46589527400325e-14"


### Run ComBat

In [14]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(SMTSD), data=sampleinfo)

batch <- factor(sampleinfo$SMGEBTCH)

expr_combat1 <- sva::ComBat(
    dat,
    batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 31327 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found177batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [15]:
# After ComBat

pca <- prcomp(t(expr_combat1), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 1.16912569905112e-05"
[1] "SMCENTER 3.21245318183572e-65"
[1] "SMNABTCH 2.13372950225428e-13"
[1] "SMGEBTCH 0.518541426587708"
[1] "SEX 0.0117217261202048"
[1] "SMTSD 1.23559042311644e-67"
[1] "DTHHRDY 3.70346779983198e-08"


In [16]:
fwrite(expr_combat1, file=paste0("data/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected.csv"), row.names=TRUE)

x being coerced from class: matrix to data.table

